In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
)

In [ ]:
# ---------- Load & prep (same first 5k articles from data_engineering) ----------
N = 5000
news_train = pd.read_csv("news.csv")
news_labels = pd.read_csv("news_topics.csv")
news_labels["cat"] = news_labels["cat"].str.strip()

articles = news_train.head(N)
ids = set(articles["id"])

# TF-IDF (text already stemmed/cleaned)
tfidf_vec = TfidfVectorizer()
X_raw = tfidf_vec.fit_transform(articles["article"])
print(f"TF-IDF: {X_raw.shape}")

# h1 top-level topics only
H1_TOPICS = ["CCAT", "ECAT", "GCAT", "MCAT"]
labels_sub = news_labels[
    (news_labels["id"].isin(ids)) & (news_labels["cat"].isin(H1_TOPICS))
]
cat_lists = labels_sub.groupby("id", sort=False)["cat"].apply(list)
labels_aligned = articles[["id"]].merge(cat_lists.rename("cats"), on="id", how="left")
labels_aligned["cats"] = labels_aligned["cats"].apply(lambda x: x if isinstance(x, list) else [])

mlb = MultiLabelBinarizer(classes=H1_TOPICS)
Y = mlb.fit_transform(labels_aligned["cats"])
print(f"Labels (h1): {Y.shape}  columns={mlb.classes_}")
print("Class counts:", dict(zip(H1_TOPICS, Y.sum(axis=0))))

In [ ]:
# ---------- PCA (TruncatedSVD for sparse matrices) ----------
N_COMPONENTS = 300
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
X = svd.fit_transform(X_raw)
print(f"Reduced: {X_raw.shape} → {X.shape}")
print(f"Explained variance (cumulative): {svd.explained_variance_ratio_.sum():.3f}")

In [ ]:
# ---------- 5-fold CV with LinearSVC (one-vs-rest for multi-label) ----------
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=42)

# Per-topic, per-fold storage
topic_metrics = {t: {"accuracy": [], "precision": [], "recall": [], "f1": []}
                 for t in H1_TOPICS}
fold_cm = {t: np.zeros((2, 2), dtype=int) for t in H1_TOPICS}

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]

    clf = OneVsRestClassifier(LinearSVC(max_iter=5000, random_state=42))
    clf.fit(X_train, Y_train)
    Y_pred = clf.predict(X_test)

    for i, topic in enumerate(H1_TOPICS):
        y_true = Y_test[:, i]
        y_pred = Y_pred[:, i]
        topic_metrics[topic]["accuracy"].append(accuracy_score(y_true, y_pred))
        topic_metrics[topic]["precision"].append(
            precision_score(y_true, y_pred, zero_division=0))
        topic_metrics[topic]["recall"].append(
            recall_score(y_true, y_pred, zero_division=0))
        topic_metrics[topic]["f1"].append(
            f1_score(y_true, y_pred, zero_division=0))
        fold_cm[topic] += confusion_matrix(y_true, y_pred, labels=[0, 1])

    print(f"Fold {fold}/{K} done")

print("All folds complete.")

In [ ]:
# ---------- Results summary ----------
rows = []
for topic in H1_TOPICS:
    m = topic_metrics[topic]
    rows.append({
        "Topic": topic,
        "Accuracy":  np.mean(m["accuracy"]),
        "Precision": np.mean(m["precision"]),
        "Recall":    np.mean(m["recall"]),
        "F1":        np.mean(m["f1"]),
    })
results_df = pd.DataFrame(rows)
print("=== SVM Baseline — H1 Topics (5-fold CV) ===\n")
print(results_df.to_string(index=False))
print(f"\nMacro-avg F1: {results_df['F1'].mean():.4f}")

# Confusion matrices (summed across folds)
print("\n=== Confusion Matrices (summed over all folds) ===")
for topic in H1_TOPICS:
    cm = fold_cm[topic]
    print(f"\n{topic}:")
    print(pd.DataFrame(cm, index=["Actual 0", "Actual 1"],
                           columns=["Pred 0", "Pred 1"]))